# Classification NBA Model

## Configuration

## Imports

In [39]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training
)
from sklearn.model_selection import KFold, cross_validate, train_test_split
from xgboost import XGBClassifier


In [40]:
from nba_ou.modeling.modeling import (
    split_latest_dates_holdout,
    make_walk_forward_last_n_seasons_splits,
    validate_time_splits,
    make_test_anchored_walk_forward_splits,
    assert_valid_time_splits,
    save_model_bundle,
    load_model_bundle
)

In [41]:
nan_threshold = 50.0
max_na_per_row = 5

## Load Data

In [42]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260317.csv"

path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)
df_stats['GAME_DATE'] = pd.to_datetime(df_stats['GAME_DATE']).dt.strftime('%Y-%m-%d')

In [43]:
import time
time.sleep(3.5*3600)

In [44]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=nan_threshold, max_na_per_row=max_na_per_row, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 11249 rows
Basic cleaning complete: 8640 rows remaining

Starting advanced column cleaning with 3240 columns

Advanced column cleaning complete: 3240 → 2137 columns (1103 removed)


Dropping NA rows for SEASON_YEAR 2017...
   Removed 0 rows with NaN values from 2017 season

Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 0 (0.0%)
  Critical columns requiring data: 4
  Columns zero-filled: 112
  Infer pairs applied: 20/136
  Remaining NaN cells: 1030992

Dropping rows with more than 5 NaN values...
Removed 4019 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (4621, 2137)


In [45]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_to_train.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1343,LAST_5_GAMES_TOTAL_POINTS_BEFORE,455,9.85,2021.0
1903,LEAGUE_GAMES_LAST_1D_BEFORE,193,4.18,2024.0
1946,total_consensus_pct_under,67,1.45,2024.0
1342,LAST_4_GAMES_TOTAL_POINTS_BEFORE,65,1.41,2021.0
1947,spread_consensus_pct_home,63,1.36,2024.0
2127,TRAVEL_RECENCY_RATIO_AWAY_2D_OVER_14D_BEFORE,42,0.91,2021.0
2126,TRAVEL_RECENCY_RATIO_HOME_2D_OVER_14D_BEFORE,17,0.37,2024.0
2078,TOP3_AVAILABILITY_EFFECT_AWAY_MAX_ABS_DIFF_FRO...,13,0.28,2022.0
2076,TOP3_AVAILABILITY_EFFECT_AWAY_MAX_ABS_TOTAL_PO...,13,0.28,2022.0
2074,TOP3_AVAILABILITY_EFFECT_AWAY_MEAN_DIFF_FROM_LINE,13,0.28,2022.0


In [46]:
BET365_LINE_COL =  "TOTAL_LINE_bet365"
# BET365_LINE_COL =  "total_bet365_line_over"

# Ensure scoring line and target exist (avoid NaN-driven undefined betting accuracy).
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [47]:
df_to_train['GAME_DATE'] = pd.to_datetime(df_to_train['GAME_DATE'])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

In [48]:
#count games per season
games_per_season = df_to_train.groupby('SEASON_YEAR').size()
print(games_per_season)

SEASON_YEAR
2021    1143
2022    1154
2023     157
2024    1233
2025     934
dtype: int64


## Train / Test

In [49]:
from sklearn.model_selection import KFold, cross_validate, cross_val_predict, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer, root_mean_squared_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.base import clone

In [50]:
df_dev, df_test_final = split_latest_dates_holdout(
    df=df_to_train,
    date_col="GAME_DATE",
    test_size=0.05,
)

print(f"Development set size: {len(df_dev)}")
print(f"Final test set size: {len(df_test_final)}")
print("Final test date range:",
      df_test_final["GAME_DATE"].min(), "->", df_test_final["GAME_DATE"].max())

Development set size: 4389
Final test set size: 232
Final test date range: 2026-02-08 00:00:00 -> 2026-03-17 00:00:00


In [51]:
TARGET_COL = "TOTAL_POINTS"

EXCLUDE_COLS = [
    "TOTAL_POINTS",
    "SEASON_YEAR",
    "GAME_DATE",
]

X_dev = df_dev.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(df_dev[TARGET_COL], errors="coerce")

X_test_final = df_test_final.drop(columns=EXCLUDE_COLS, errors="ignore")
y_test_final = pd.to_numeric(df_test_final[TARGET_COL], errors="coerce")

print(f"X_dev shape: {X_dev.shape}")
print(f"X_test_final shape: {X_test_final.shape}")

X_dev shape: (4389, 2134)
X_test_final shape: (232, 2134)


In [52]:
from nba_ou.modeling.scorers import OverUnderScorerTotalPoints, over_under_betting_accuracy_total_points, evaluate_total_points_thresholds
    
ou_scorer = OverUnderScorerTotalPoints(BET365_LINE_COL)

scoring = {
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
    "OU_Betting_Accuracy": ou_scorer,
}

def print_metrics(cv_results):
    for sc in scoring.keys():
        train_key = f"train_{sc}"
        test_key = f"test_{sc}"

        train_val = cv_results[train_key].mean()
        test_val = cv_results[test_key].mean()

        if sc in {"MSE", "RMSE", "MAE"}:
            train_val = -train_val
            test_val = -test_val

        if sc == "OU_Betting_Accuracy":
            print(f"Train {sc}: {train_val:.2%}")
            print(f"Validation {sc}: {test_val:.2%}")
        else:
            print(f"Train {sc}: {train_val:.5f}")
            print(f"Validation {sc}: {test_val:.5f}")
        print()

In [71]:
train_games = 2500

splits, fold_info = make_test_anchored_walk_forward_splits(
    df=df_dev,
    date_col="GAME_DATE",
    season_col="SEASON_YEAR",
    test_games=30,
    step_games_between_tests=50,
    train_games=train_games,
    min_train_games=2000,
    max_folds=12,
    verbose=1,
)


Created 12 test-anchored walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date  test_season
    1           2500            33       2022-03-07     2025-03-08      2025-03-09    2025-03-12         2024
    2           2500            36       2022-03-20     2025-03-19      2025-03-20    2025-03-24         2024
    3           2500            32       2022-04-01     2025-03-31      2025-04-01    2025-04-04         2024
    4           2500            31       2022-04-22     2025-04-11      2025-04-13    2025-04-25         2024
    5           2500            32       2022-11-04     2025-06-22      2025-10-27    2025-11-03         2025
    6           2500            30       2022-11-18     2025-11-10      2025-11-11    2025-11-14         2025
    7           2500            30       2022-11-30     2025-11-22      2025-11-23    2025-11-26         2025
    8           2500            31       2022-12-12     2025-12-03      2025

In [54]:
season_bl = DummyRegressor(strategy="mean")

cv_results = cross_validate(
    season_bl,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("DummyRegressor baseline")
print_metrics(cv_results)

DummyRegressor baseline
Train MSE: 377.15287
Validation MSE: 368.65402

Train RMSE: 19.42023
Validation RMSE: 18.98152

Train MAE: 15.49299
Validation MAE: 15.25653

Train R2: 0.00000
Validation R2: -0.07379

Train OU_Betting_Accuracy: 49.91%
Validation OU_Betting_Accuracy: 52.41%



In [55]:
lr = LinearRegression()

cv_results = cross_validate(
    lr,
    X_dev.fillna(0),   # LR cannot handle NaNs
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print("Linear Regression")
print_metrics(cv_results)

Linear Regression
Train MSE: 117.79596
Validation MSE: 7473334.75324

Train RMSE: 10.85272
Validation RMSE: 858.38934

Train MAE: 8.44564
Validation MAE: 189.11208

Train R2: 0.68765
Validation R2: -11125.13202

Train OU_Betting_Accuracy: 78.72%
Validation OU_Betting_Accuracy: 45.46%



In [56]:
xgb_reg = XGBRegressor(
    max_depth=4,
    learning_rate=0.057,
    n_estimators=75,
    subsample=0.8,
    colsample_bytree=0.86,
    reg_alpha=0.57,
    reg_lambda=1.78,
    min_child_weight=5.48,
    gamma=1.77,
    n_jobs=-2,          # choose your CPU count
    random_state=16,
)

cv_results = cross_validate(
    xgb_reg,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,          # leave at 1 because XGB already parallelizes internally
)

print("XGBoost")
print_metrics(cv_results)

XGBoost
Train MSE: 169.27506
Validation MSE: 309.65849

Train RMSE: 13.01035
Validation RMSE: 17.38062

Train MAE: 10.34766
Validation MAE: 14.05990

Train R2: 0.55118
Validation R2: 0.08979

Train OU_Betting_Accuracy: 83.03%
Validation OU_Betting_Accuracy: 54.42%



In [57]:
xgb_reg.fit(X_dev, y_dev)

y_pred_test_total = xgb_reg.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_test_final,
    y_pred_test_total,
    betting_line,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

Final test metrics
MSE: 297.42926
RMSE: 17.24614
MAE: 13.56271
OU_Betting_Accuracy: 54.87%


In [58]:
results_df, y_pred_test_total = evaluate_total_points_thresholds(
    model=xgb_reg,
    X_test=X_test_final,
    y_test_total=y_test_final,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_betting_accuracy": "{:.2%}"}
    )
)

,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,0,232,100.0%,54.87%
1,1,155,66.8%,52.98%
2,2,105,45.3%,56.86%
3,3,51,22.0%,56.00%
4,4,17,7.3%,56.25%
5,5,6,2.6%,50.00%
6,6,2,0.9%,50.00%
7,7,1,0.4%,100.00%
8,8,0,0.0%,nan%
9,9,0,0.0%,nan%


# OPTUNA

In [59]:
from nba_ou.modeling.optuna_total_points import (
    tune_xgb_total_points_optuna,
    summarize_optuna_trials,
    fit_best_xgb_total_points,
)

study = tune_xgb_total_points_optuna(
    X=X_dev,
    y=y_dev,
    splits=splits,
    line_col=BET365_LINE_COL,
    n_trials=80,
    timeout=4.5 * 3600,
    objective_name="reg:squarederror",   # second run: reg:pseudohubererror
    study_name="xgb_total_points_mae",
)

print("Best CV MAE:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")

print("\nSecondary metrics from best trial:")
print("Mean RMSE:", study.best_trial.user_attrs.get("mean_rmse"))
print("Mean R2:", study.best_trial.user_attrs.get("mean_r2"))
print("Mean OU accuracy:", study.best_trial.user_attrs.get("mean_ou_acc"))
print("Mean best_iteration:", study.best_trial.user_attrs.get("mean_best_iteration"))

trials_df = summarize_optuna_trials(study)
display(
    trials_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
        }
    )
)

[I 2026-03-19 02:42:32,704] A new study created in memory with name: xgb_total_points_mae


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-19 02:46:47,858] Trial 0 finished with value: 13.616326590120872 and parameters: {'max_depth': 2, 'min_child_weight': 18.346704707583235, 'gamma': 1.6970342240854253, 'subsample': 0.5682407800531226, 'colsample_bytree': 0.5123279759064977, 'learning_rate': 0.011926786034588454, 'reg_alpha': 1.8771791376898666, 'reg_lambda': 1.897469395521307}. Best is trial 0 with value: 13.616326590120872.
[I 2026-03-19 02:52:55,890] Trial 1 finished with value: 13.577119661058722 and parameters: {'max_depth': 2, 'min_child_weight': 51.819268381786635, 'gamma': 1.7346760026809933, 'subsample': 0.5811969357559948, 'colsample_bytree': 0.675188230006178, 'learning_rate': 0.010426963054371881, 'reg_alpha': 0.06701717253228541, 'reg_lambda': 3.152289132591451}. Best is trial 1 with value: 13.577119661058722.
[I 2026-03-19 02:57:34,070] Trial 2 finished with value: 13.57239681311654 and parameters: {'max_depth': 4, 'min_child_weight': 15.848753157115912, 'gamma': 0.7236802167715846, 'subsample': 

,trial,value_mae,mean_rmse,mean_r2,mean_ou_acc,mean_best_iteration,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda
0,57,13.3848,16.8287,0.1533,58.54%,122,3,34.707341,1.962504,0.937508,0.777493,0.041247,0.038111,25.791156
1,14,13.3905,16.8530,0.1504,56.95%,59,3,59.787549,2.207130,0.887906,0.741546,0.053857,0.203085,25.497831
2,11,13.3933,16.9099,0.1466,55.94%,71,3,29.350210,2.942417,0.877483,0.753025,0.057036,0.014402,22.642312
3,51,13.3972,16.8801,0.1480,57.62%,82,3,48.820974,1.752442,0.880745,0.755574,0.059905,0.013921,29.908175
4,34,13.4005,16.8932,0.1460,61.45%,57,3,32.902486,1.565812,0.888214,0.673435,0.053874,0.053496,20.697349
5,19,13.4161,16.9281,0.1428,59.31%,45,3,59.979674,2.093913,0.891740,0.711667,0.059614,0.033913,25.140282
6,62,13.4307,16.8789,0.1487,59.11%,82,3,25.261257,2.336431,0.946269,0.754118,0.044825,0.078812,16.752388
7,58,13.4317,16.9199,0.1439,56.25%,108,3,20.511722,1.978027,0.948813,0.777684,0.041143,0.036015,25.325115
8,52,13.4333,16.9622,0.1406,59.29%,69,3,42.131986,1.740645,0.872842,0.735592,0.059847,0.014564,28.600061
9,32,13.4459,16.9326,0.1418,55.64%,59,3,51.021789,1.846531,0.893686,0.734060,0.058740,0.011729,29.120082


In [78]:
total_df = df_dev.tail(train_games+500)
total_df = df_dev[df_dev["SEASON_YEAR"] >= 2023].copy()

In [79]:
X_dev = total_df.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(total_df[TARGET_COL], errors="coerce")

In [80]:
best_model = fit_best_xgb_total_points(
    X_dev=X_dev,
    y_dev=y_dev,
    study=study,
    objective_name="reg:squarederror",
)

y_pred_test_total = best_model.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

Final test metrics
MSE: 305.54660
RMSE: 17.47989
MAE: 13.65518
OU_Betting_Accuracy: 51.33%


In [61]:
# Limit to Season year 2023 onwards
train_data = df_to_train[df_to_train["SEASON_YEAR"] >= 2023].copy()

In [62]:
from nba_ou.modeling.modeling import ModelBundleMetadata, ModelInfo, TrainingMetrics

X_full = train_data.drop(columns=EXCLUDE_COLS, errors="ignore")
y_full = pd.to_numeric(train_data[TARGET_COL], errors="coerce")


production_model = fit_best_xgb_total_points(
    X_dev=X_full,
    y_dev=y_full,
    study=study,
    objective_name="reg:squarederror",
)

latest_training_date = pd.to_datetime(train_data["GAME_DATE"]).max()
model_version = latest_training_date.strftime("%d_%m_%y")
model_name = f"three_seasons_xgb_total_points_{model_version}"

metadata = ModelBundleMetadata(
    model_info=ModelInfo(
        name=model_name,
        model_version=model_version,
        model_type="three_seasons_total_points",
        prediction_source="three_seasons_xgb_total_points",
        training_code_tag="1.0",
    ),
    training_metrics=TrainingMetrics(
        best_params=study.best_trial.params,
        mean_best_iteration=study.best_trial.user_attrs.get("mean_best_iteration"),
        cv_mae=float(study.best_value),
        cv_rmse=study.best_trial.user_attrs.get("mean_rmse"),
        cv_ou_acc=study.best_trial.user_attrs.get("mean_ou_acc"),
        final_test_mae=float(mae),
        final_test_rmse=float(rmse),
        final_test_ou_acc=float(ou_acc),
        nan_threshold=nan_threshold,
        max_na_per_row=max_na_per_row,
        train_date_min=train_data["GAME_DATE"].min().to_pydatetime(),
        train_date_max=train_data["GAME_DATE"].max().to_pydatetime(),
    ),
)

model_path, meta_path = save_model_bundle(
    model=production_model,
    feature_names=list(X_full.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/3_seasons/",
    metadata=metadata,
)

print(f"Production model trained on {len(X_full)} rows using fixed n_estimators from mean_best_iteration.")
print("Saved model :", model_path)
print("Saved metadata:", meta_path)


Production model trained on 2324 rows using fixed n_estimators from mean_best_iteration.
Saved model : /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/3_seasons/three_seasons_xgb_total_points_17_03_26.json
Saved metadata: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/3_seasons/three_seasons_xgb_total_points_17_03_26.meta.json
